# tp-03 特徴量の ablation（どの特徴量・どの対処を採用するか）

**実行場所: ホスト PC（stg）。** 先に tp-01・tp-02 を通すこと（`features/train_table.parquet`、`feature_families.json`）。

## 目的
tp-02 で作った履歴特徴量 127 列（A〜H の 11 ファミリー）のうち、**どれを採用するか**と、**時代を見分ける特徴量への対処**を、時系列 CV の予測精度で決める。

## 条件（固定）
- モデル: LightGBM（`n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.8, colsample_bytree=0.7, reg_lambda=5`）、3 seed。
- CV: valid 2022 / 2023 / 2024 の expanding window（purge 7 日 + embargo 3 日）。**test 2025 は tp-06 まで使わない。**
- 主指標: p_win の LogLoss。AUC、レース内 softmax の勝ち馬の負の対数尤度（race_nll）を併記。最終候補だけ p_top2・p_top3 も見る。
- オッズ・人気は使わない。2026 年は使わない。

## 採否の基準
「共通の全 fold で悪化せず、平均が改善し、改善幅が max(0.0001, 2×seed 間のばらつき) 以上」。相関が 0.9 以上の特徴量は代表 1 つを残す案も、残さない案と比べて**良い方を採用**する。

## 実験の段階
| 段階 | 内容 |
|---|---|
| 0 | B0（LayerA の既存列）と、運用で使えない可能性のある `speed_*` を除いた B0′ |
| 1 | B0 + 各ファミリー（単独） |
| 2 | B0 + F3 + 各ファミリー（累積） |
| 3 | 全部 − 各ファミリー（leave-one-out） |
| 4 | 時代ずれ対策: F5 除外／件数系除外／クラス・相手関係系の年内標準化（直前 365 日・as-of）／2020〜2021 年を学習から外す／2020 年の重み |
| 5 | 展開（F2）と RPCI（G）の比較・併用 |
| 6 | 採用候補の確定 → 相関の整理（する／しない）を比較 → 最終候補を p_win・p_top2・p_top3 で評価、adversarial validation |

途中で止まっても、`ablation/ablation_results.csv` の結果を再利用して続きから再開できる。

## ベースの 2 通り（`TP03_BASE`）
B0 の `speed_*`（タイム指数）は、**開催前に取得できるか未確認**（SLA 監査 G11: 開催前は 0 件）。使えないと運用で動かないので、**speed を含む（参考）／含まない（運用の前提）の両方**で同じ手順を回し、結果を併記する。出力は `ablation/`（speed）と `ablation_nospeed/`。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.project_env import load_project_dotenv

load_project_dotenv()
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import json, subprocess, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ablation as ab
from src.pipeline.models.compute import detect_compute
from src.pipeline.models.time_split import yearly_expanding_folds

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
FEAT = WORK / "features"
# TP03_BASE=speed: B0 に speed_*（当該レースのタイム指数ページ由来）を含む／nospeed: 含まない。
# speed_* は開催前に取得できるかが未確認（SLA 監査 G11）なので、運用で使える前提の nospeed を主、speed を参考として両方評価する。
BASE_KEY = os.environ.get("TP03_BASE", "speed")
assert BASE_KEY in ("speed", "nospeed")
OUT = WORK / ("ablation" if BASE_KEY == "speed" else "ablation_nospeed")
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = OUT / "ablation_results.csv"
COMPUTE = detect_compute()
THREADS = COMPUTE.threads
SEEDS = [0, 1, 2]

MLFLOW_URI = require_tracking_server()
mlflow.set_experiment("keiba-tp-ablation")
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
print("MLflow:", MLFLOW_URI, "| git:", GIT_COMMIT, "| threads:", THREADS, "| seeds:", SEEDS, "| base:", BASE_KEY)

In [ ]:
df = ab.load_table(FEAT / "train_table.parquet")
fam = json.loads((FEAT / "feature_families.json").read_text(encoding="utf-8"))
FAMS = ["A", "B", "C", "D", "E", "F1", "F2", "F3", "F5", "G", "H"]
assert all(f in fam for f in FAMS), "feature_families.json にファミリーが足りません"
for f in FAMS:
    miss = [c for c in fam[f] if c not in df.columns]
    assert not miss, (f, miss[:3])
B0 = ab.b0_columns(include_speed=(BASE_KEY == "speed"))
B0P = ab.b0_columns(include_speed=False)
miss = [c for c in B0 if c not in df.columns]
assert not miss, f"B0 の列が学習用テーブルにありません: {miss}"
ALL_FEATS = [c for f in FAMS for c in fam[f]]
print(f"行 {len(df):,} / レース {df['race_id'].nunique():,} / B0 {len(B0)} 列 / 履歴特徴量 {len(ALL_FEATS)} 列 / 期間 {df['date'].min():%Y-%m-%d}〜{df['date'].max():%Y-%m-%d}")
print({f: len(fam[f]) for f in FAMS})
print("ラベル率 win/top2/top3:", [round(float(df[c].mean()), 4) for c in ("is_win", "is_top2", "is_top3")])
folds = list(yearly_expanding_folds(df))
for f in folds:
    print(f.fold_id, "学習", len(f.train_index), "検証", len(f.valid_index), "学習終端(含まず)", f.train_end.date())
assert len(set(df["race_id"].iloc[folds[0].train_index]) & set(df["race_id"].iloc[folds[0].valid_index])) == 0

In [ ]:
# 結果の保存と再開
ALL_ROWS = pd.read_csv(RESULTS).to_dict("records") if RESULTS.exists() else []


def run(spec, targets=("win",), seeds=SEEDS):
    """spec を評価して結果を保存する。同じ spec・target の結果が揃っていれば再利用する。"""
    global ALL_ROWS
    have = {(r["key"], r["target"]) for r in ALL_ROWS}
    need = [t for t in targets if (spec.key(), t) not in have]
    if need:
        t0 = time.time()
        rows = ab.evaluate(df, folds, spec, seeds, THREADS, targets=need)
        ALL_ROWS += rows
        pd.DataFrame(ALL_ROWS).to_csv(RESULTS, index=False)
        m = np.mean([r["logloss"] for r in rows if r["target"] == "win"]) if any(r["target"] == "win" for r in rows) else float("nan")
        print(f"  {spec.name:44s} win LogLoss {m:.5f}  ({time.time() - t0:.0f}s, {len(spec.feats)}列)")
    return spec


def table(names=None):
    r = pd.DataFrame(ALL_ROWS)
    s = ab.summarize(r)
    return s if names is None else s.loc[[n for n in names if n in s.index]]


def vs(base, names):
    s = table()
    c = ab.compare_to_base(s.loc[[base] + [n for n in names if n in s.index]], base)
    return c

## 段階 0: ベースライン B0 と B0′（`speed_*` なし）

In [ ]:
run(ab.Spec("B0", B0))
run(ab.Spec("B0′(speed なし)", B0P))
display(table(["B0", "B0′(speed なし)"]).round(5))
print("B0 の fold 別 LogLoss と seed 間のばらつき（これが採用基準の下限になる）")

## 段階 1: B0 + 各ファミリー（単独）
各ファミリーの単独の効果。

In [ ]:
S1 = [run(ab.Spec(f"B0+{f}", B0 + fam[f])) for f in FAMS]
c1 = vs("B0", [s.name for s in S1])
display(c1.round(5))

## 段階 2: B0 + F3 + 各ファミリー（累積）
最も効く F3（Elo）に上乗せしたときの追加効果。

In [ ]:
run(ab.Spec("B0+F3", B0 + fam["F3"]))
S2 = [run(ab.Spec(f"B0+F3+{f}", B0 + fam["F3"] + fam[f])) for f in FAMS if f != "F3"]
c2 = vs("B0+F3", [s.name for s in S2])
display(c2.round(5))

## 段階 3: 全部 − 各ファミリー（leave-one-out）
冗長なブロックの検出。外して悪化するなら必要、変わらない・良くなるなら冗長。

In [ ]:
ALLSPEC = run(ab.Spec("ALL", B0 + ALL_FEATS))
S3 = [run(ab.Spec(f"ALL−{f}", B0 + [c for c in ALL_FEATS if c not in fam[f]])) for f in FAMS]
c3 = vs("ALL", [s.name for s in S3])
display(c3.round(5))
print("Δ平均 が正 = そのファミリーを外すと悪化（必要）／負 = 外した方が良い（冗長か有害）")

## 段階 4: 時代ずれ対策
adversarial validation の AUC は「分布差の大きさ」の目安で、合否基準ではない。**LogLoss を悪化させずに**下げられるものを探す。

In [ ]:
COUNT = [c for c in ab.COUNT_COLS if c in ALL_FEATS]
ERA = [c for c in ab.ERA_CLASS_COLS if c in ALL_FEATS]
t0 = time.time()
Z = ab.asof_standardize(df, ERA, window_days=365, min_n=2000)
print(f"年内標準化（直前365日・as-of）{len(ERA)} 列を計算 ({time.time() - t0:.0f}s) | 欠損率", float(Z.isna().mean().mean().round(3)))
no_f5 = [c for c in ALL_FEATS if c not in fam["F5"]]
no_cnt = [c for c in ALL_FEATS if c not in COUNT]
no_f5_cnt = [c for c in no_f5 if c not in COUNT]
S4 = [
    run(ab.Spec("ALL−F5", B0 + no_f5)),
    run(ab.Spec("ALL−件数系", B0 + no_cnt)),
    run(ab.Spec("ALL−F5−件数系", B0 + no_f5_cnt)),
    run(ab.Spec("ALL+年内標準化(クラス系)", B0 + ALL_FEATS, replace={c: Z[c] for c in ERA})),
    run(ab.Spec("ALL−F5−件数系+年内標準化", B0 + no_f5_cnt, replace={c: Z[c] for c in ERA})),
    run(ab.Spec("ALL・2022年以降で学習", B0 + ALL_FEATS, min_train_year=2022)),
    run(ab.Spec("ALL・2020年の重み0.5", B0 + ALL_FEATS, weights={2020: 0.5})),
]
c4 = vs("ALL", [s.name for s in S4])
display(c4.round(5))
print("注: 『2022年以降で学習』は fold 2022 が評価できないため、共通の fold（2023・2024）だけで ALL と比べている")

## 段階 5: 展開（F2）と RPCI（G）の比較・併用
前半3F−後半3Fの差で見る F2 と、距離で正規化した RPCI で見る G のどちらが効くか、両方入れて冗長でないか。

In [ ]:
S5 = [
    run(ab.Spec("B0+F3+F2", B0 + fam["F3"] + fam["F2"])),
    run(ab.Spec("B0+F3+G", B0 + fam["F3"] + fam["G"])),
    run(ab.Spec("B0+F3+F2+G", B0 + fam["F3"] + fam["F2"] + fam["G"])),
    run(ab.Spec("B0+F3+H", B0 + fam["F3"] + fam["H"])),
    run(ab.Spec("B0+F3+H+G", B0 + fam["F3"] + fam["H"] + fam["G"])),
]
display(vs("B0+F3", [s.name for s in S5]).round(5))

## 段階 6: 採用候補の確定
ファミリーは次の規則で採否を決める。
1. 段階 2（B0 + F3 への上乗せ）で採用基準を満たすファミリーを採用。
2. 満たさなくても、段階 3（全部から外すと悪化）で外した場合の Δ平均 が `+max(0.0001, 2×seed ばらつき)` 以上なら採用（他と組み合わせて効くため）。
3. F3 は基準の土台なので採用。

In [ ]:
s = table()
base_sd = float(s.loc["ALL", "seed_sd"])
thr = max(0.0001, 2 * (0.0 if np.isnan(base_sd) else base_sd))
adopt, why = ["F3"], {"F3": "土台（単独で最も効く）"}
for f in FAMS:
    if f == "F3":
        continue
    cum = c2.loc[f"B0+F3+{f}"]
    loo = c3.loc[f"ALL−{f}"]
    if bool(cum["採用基準を満たす"]):
        adopt.append(f); why[f] = f"累積で採用基準を満たす（Δ平均 {cum['Δ平均']:+.5f}）"
    elif float(loo["Δ平均"]) >= thr:
        adopt.append(f); why[f] = f"累積では基準未満だが、外すと悪化（Δ平均 {loo['Δ平均']:+.5f}）"
    else:
        why[f] = f"不採用（累積 Δ平均 {cum['Δ平均']:+.5f}／外した場合 {loo['Δ平均']:+.5f}）"
ADOPT_FAMS = [f for f in FAMS if f in adopt]
print("採用ファミリー:", ADOPT_FAMS)
for f in FAMS:
    print(f"  {f:3s}", why[f])
ADOPT_FEATS = [c for f in ADOPT_FAMS for c in fam[f]]

In [ ]:
# 時代ずれ対策のうち採用基準を満たしたもの（段階 4）。AUC が下がっても LogLoss が悪化するものは採らない
era_pass = [n for n in c4.index if bool(c4.loc[n, "採用基準を満たす"])]
era_nohurt = [n for n in c4.index if bool(c4.loc[n, "全foldで改善"]) or float(c4.loc[n, "Δ平均"]) <= 0.00005]
print("採用基準を満たした対策:", era_pass)
print("悪化しない（Δ平均 ≤ +0.00005 か全 fold 改善）対策:", era_nohurt)

In [ ]:
# 採用候補を比べる: ADOPTED／ADOPTED＋相関の整理／ALL。良い方を採用する
run(ab.Spec("ADOPTED", B0 + ADOPT_FEATS))
# 重要度順（最後の fold の全学習データでの gain）で並べ、相関 0.9 以上は代表 1 つにする案
import lightgbm as lgb
fold = folds[-1]
Xa = df[B0 + ADOPT_FEATS]
cats = [c for c in Xa.columns if str(Xa[c].dtype) == "category"]
bst = lgb.train(dict(ab.LGB_PARAMS, seed=0, num_threads=THREADS), lgb.Dataset(Xa.iloc[fold.train_index], label=df["is_win"].to_numpy()[fold.train_index], categorical_feature=cats or "auto"), num_boost_round=ab.N_ROUNDS)
imp = pd.Series(bst.feature_importance("gain"), index=Xa.columns).sort_values(ascending=False)
kept, dropped = ab.drop_correlated(df.iloc[fold.train_index], ADOPT_FEATS, order=list(imp.index), threshold=0.9)
print(f"相関の整理: {len(ADOPT_FEATS)} → {len(kept)} 列（{len(dropped)} 列を代表に寄せた）")
run(ab.Spec("ADOPTED・相関整理", B0 + kept))
display(vs("B0", ["ALL", "ADOPTED", "ADOPTED・相関整理"]).round(5))

In [ ]:
# 時代ずれ対策を ADOPTED に足した案（段階 4 で LogLoss を悪化させなかったもの）も比べる
extra = []
if any("年内標準化" in n for n in era_nohurt):
    ERA_A = [c for c in ERA if c in ADOPT_FEATS]
    extra.append(run(ab.Spec("ADOPTED+年内標準化", B0 + ADOPT_FEATS, replace={c: Z[c] for c in ERA_A})))
if any("件数系" in n for n in era_nohurt):
    extra.append(run(ab.Spec("ADOPTED−件数系", B0 + [c for c in ADOPT_FEATS if c not in COUNT])))
if any("F5" in n for n in era_nohurt) and "F5" in ADOPT_FAMS:
    extra.append(run(ab.Spec("ADOPTED−F5", B0 + [c for c in ADOPT_FEATS if c not in fam["F5"]])))
cands = ["ALL", "ADOPTED", "ADOPTED・相関整理"] + [s.name for s in extra]
final_cmp = vs("B0", cands)
display(final_cmp.round(5))
s = table()
best = min(cands, key=lambda n: float(s.loc[n, "mean"]))
print("最終候補（平均 LogLoss が最小）:", best, f"{float(s.loc[best, 'mean']):.5f}", "| B0:", f"{float(s.loc['B0', 'mean']):.5f}")

## 最終候補の評価（p_win・p_top2・p_top3）と adversarial validation

In [ ]:
SPECS = {sp.name: sp for sp in [ab.Spec("B0", B0), ab.Spec("ALL", B0 + ALL_FEATS), ab.Spec("ADOPTED", B0 + ADOPT_FEATS),
                                ab.Spec("ADOPTED・相関整理", B0 + kept)] + extra}
for n in dict.fromkeys(["B0", best]):
    run(SPECS[n], targets=("win", "top2", "top3"))
r = pd.DataFrame(ALL_ROWS)
rows = []
for n in dict.fromkeys(["B0", best]):
    x = r[r["exp"] == n]
    rows.append({"構成": n, **{f"{t} LogLoss": float(x[x["target"] == t]["logloss"].mean()) for t in ("win", "top2", "top3")},
                 "win AUC": float(x[x["target"] == "win"]["auc"].mean()), "win race_nll": float(x[x["target"] == "win"]["race_nll"].mean())})
final_table = pd.DataFrame(rows).set_index("構成")
display(final_table.round(5))

In [ ]:
yrs = df["year"].to_numpy()
tr_m, va_m, te_m = yrs <= 2023, yrs == 2024, yrs == 2025
FINAL_FEATS = (B0 + kept) if best == "ADOPTED・相関整理" else (B0 + ADOPT_FEATS if best in ("ADOPTED", "ADOPTED+年内標準化", "ADOPTED−件数系", "ADOPTED−F5") else B0 + ALL_FEATS)
if best == "ADOPTED−件数系": FINAL_FEATS = [c for c in FINAL_FEATS if c not in COUNT]
if best == "ADOPTED−F5": FINAL_FEATS = [c for c in FINAL_FEATS if c not in fam["F5"]]
adv = {}
for label, feats in {"B0": B0, "ALL": B0 + ALL_FEATS, best: FINAL_FEATS}.items():
    for nm, (a, b) in {"train_vs_valid": (tr_m, va_m), "train_vs_test": (tr_m, te_m)}.items():
        auc, top = ab.adversarial_auc(df, feats, a, b, THREADS)
        adv[f"{label}|{nm}"] = {"auc": round(auc, 3), "上位": [f for f, _ in top[:5]]}
        print(f"{label:22s} {nm:15s} AUC={auc:.3f} 上位={[f for f, _ in top[:5]]}")

In [ ]:
adopted = {
    "base": BASE_KEY, "git_commit": GIT_COMMIT, "best": best, "families": ADOPT_FAMS, "family_reasons": why,
    "features": [c for c in FINAL_FEATS if c not in B0], "b0_features": B0,
    "dedup_dropped": dropped, "era_countermeasures": {"通過": era_pass, "悪化しない": era_nohurt},
    "standardized_class_cols": ERA if "年内標準化" in best else [],
    "threshold": thr,
}
(OUT / "adopted_features.json").write_text(json.dumps(adopted, ensure_ascii=False, indent=1), encoding="utf-8")
final_table.to_csv(OUT / "final_candidates.csv")
pd.DataFrame(ALL_ROWS).to_csv(RESULTS, index=False)
table().to_csv(OUT / "summary_by_experiment.csv")
with mlflow.start_run(run_name=f"tp-03-feature-ablation-{BASE_KEY}"):
    mlflow.log_params({"base": BASE_KEY, "git_commit": GIT_COMMIT, "seeds": SEEDS, "n_rounds": ab.N_ROUNDS, "best": best, "adopted_families": ",".join(ADOPT_FAMS)})
    s = table()
    # MLflow の指標名は英数字・_ - . 空白のみ。日本語・記号を含む実験名は ASCII の別名で記録する
    for ascii_name, n in {"B0": "B0", "ALL": "ALL", "ADOPTED": "ADOPTED", "ADOPTED_dedup": "ADOPTED・相関整理", "BEST": best}.items():
        if n in s.index:
            mlflow.log_metric(f"logloss_mean.{ascii_name}", float(s.loc[n, "mean"]))
    ascii_of = {"B0": "B0", "ALL": "ALL", best: "BEST"}
    for k, v in adv.items():
        label, nm = k.split("|")
        mlflow.log_metric(f"adv_auc.{ascii_of.get(label, 'X')}.{nm}", v["auc"])
    mlflow.log_artifacts(str(OUT))
print("保存:", OUT)
print("採用:", ADOPT_FAMS, "| 最終候補:", best, "| 特徴量数(履歴):", len(adopted["features"]))